# Mustang: user resampling

This notebook prepares Mustang workloads for the Greenfilling experiments. It increases the share of long and wide jobs by changing the probability of selecting each source user.

A selected user contributes all of their replay jobs with the original attributes and submission times. The original context jobs remain fixed. The method adapts the user-based resampling method of [Zakay and Feitelson (ICPE 2013)](https://research.spec.org/icpe_proceedings/2013/proceedings/p149.pdf) to the four-week Mustang slack and stress windows. It does not model how users respond to congestion.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_repository_root(start):
    for candidate in [start, *start.parents]:
        helper = candidate / "workloads" / "resampling" / "mustang_user_resampling_analysis.py"
        if helper.is_file():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from inside the experimental-artifacts checkout."
    )


ROOT = find_repository_root(Path.cwd().resolve())
WORKLOAD_DIR = ROOT / "workloads/original"
WORKLOAD_ROOT = ROOT / "workloads"
RESAMPLING_DIR = WORKLOAD_ROOT / "resampling"
ARTIFACT_DIR = RESAMPLING_DIR / "artifacts"
FIGURE_DIR = RESAMPLING_DIR / "figures"
if str(RESAMPLING_DIR) not in sys.path:
    sys.path.insert(0, str(RESAMPLING_DIR))

import mustang_user_resampling_analysis as mura

pd.set_option("display.max_columns", 120)
mura.configure_plot_style()

ROOT, ARTIFACT_DIR, FIGURE_DIR

## Configure the collection

The configuration fixes the source windows, sampling weights, random seeds, and output paths. There are 21 workloads per regime: the original, five replicas with uniform sampling, and five replicas for each of three heavy-user weights.

`GENERATE_WORKLOADS = False` loads the existing collection. The loading step checks that its parameters match those selected in this notebook. To create a different collection, choose a new `ARTIFACT_DIR`, set `GENERATE_WORKLOADS = True`, and run the notebook from this cell. Generation writes files in `ARTIFACT_DIR`.

In [ ]:
GENERATE_WORKLOADS = False

SOURCE_CSV = WORKLOAD_ROOT / "sources" / "mustang.csv"
REGIMES = {
    "slack": ("2015-04-13", "2015-05-11"),
    "stress": ("2013-01-07", "2013-02-04"),
}
SCENARIO_WEIGHTS = {
    "random": 1.0,
    "heavy_2x": 2.0,
    "heavy_4x": 4.0,
    "heavy_8x": 8.0,
}
SEEDS = [20260926, 20260927, 20260928, 20260929, 20260930]

planned_synthetic = len(REGIMES) * len(SCENARIO_WEIGHTS) * len(SEEDS)
generation_scope = pd.DataFrame(
    {
        "Value": [
            "; ".join(f"{name}: {start} to {end}" for name, (start, end) in REGIMES.items()),
            ", ".join(f"{name}: {weight:g}x" for name, weight in SCENARIO_WEIGHTS.items()),
            ", ".join(map(str, SEEDS)),
            f"{len(REGIMES)} baselines + {planned_synthetic} synthetic workloads",
        ]
    },
    index=["Replay windows", "Sampling weights", "Seeds", "Planned collection"],
)
display(generation_scope)
print(f"Collection: {ARTIFACT_DIR.relative_to(ROOT)}")
print(f"Figures: {FIGURE_DIR.relative_to(ROOT)}")

## Validate the source data

Before generating workloads, the notebook reloads the full Mustang CSV and applies the same eligibility rules used by the existing Mustang workflow:

- the job has a valid user ID and valid submission and execution times;
- `0 < node_count <= 1600`;
- real runtime is at least one second; and
- requested walltime is at least the real runtime.

A row may fail more than one rule, so the diagnostic reason counts do not need to add up to the total number of excluded rows. The notebook then reconstructs the original slack and stress extracts from the eligible rows. This establishes the baselines that every generated workload is checked against.

In [ ]:
source, hygiene = mura.load_source(SOURCE_CSV, REGIMES)
display(mura.format_hygiene_summary(hygiene))


## Inspect the original job shapes

First inspect what the two source regimes contain. The upper plots show the runtime and node-count distributions. The lower plots show how runtime and node count occur together.

The two-hour and 160-node reference lines are descriptive: they separate long and wide jobs in the later comparison plots. They do **not** define heavy users. Heavy-user classification is based on job node-hours and is calibrated in the next section.

In [ ]:
source_pools = {
    regime: mura.source_replay_pool(source, bounds)
    for regime, bounds in REGIMES.items()
}
source_job_summary = mura.summarize_source_job_distributions(source_pools)
display(mura.format_source_job_summary(source_job_summary))

source_job_figure = mura.plot_source_job_distributions(
    source_pools, figure_dir=FIGURE_DIR
)
display(source_job_figure)
plt.close(source_job_figure)
print(f"Saved PDF and 600 dpi PNG in {FIGURE_DIR.relative_to(ROOT)}.")

## Define heavy users in three steps

A source user is heavy only if they repeatedly submit high-node-hour jobs. The rule therefore needs three parameters:

1. `HIGH_JOB_THRESHOLD_QUANTILE`: which jobs count as high node-hours;
2. `MIN_JOBS_FOR_HEAVY`: how much user history is required; and
3. `HEAVY_USER_THRESHOLD_QUANTILE`: how large a high-job share makes an eligible user heavy.

Each parameter gets one calibration plot. The code cell immediately after the plot sets the current value and reports its exact effect.

### 1. Define a high-node-hour job

For each replay job, compute

$$
\text{job node-hours} = \text{node count} \times \frac{\text{runtime in seconds}}{3600}.
$$

At each candidate percentile, the chart shows the composition of the jobs above the corresponding node-hour threshold. Each vertical slice adds to 100% and separates jobs that are long, wide, both, or neither according to the descriptive reference lines.

Moving right makes the high-job definition stricter. The plot is a decision aid; the next cell sets the percentile actually used by the generator.

In [ ]:
job_threshold_figure = mura.plot_job_threshold_decision(
    source_pools, figure_dir=FIGURE_DIR
)
display(job_threshold_figure)
plt.close(job_threshold_figure)


Set `HIGH_JOB_THRESHOLD_QUANTILE` below. A job is high when its node-hours are **strictly greater** than the regime-specific quantile.

After changing the value, inspect the table to confirm the resulting threshold, number of high jobs, and long/wide composition in both regimes.

In [ ]:
# Classification choice 1: job-level work cutoff.
HIGH_JOB_THRESHOLD_QUANTILE = 0.70

job_cutoff_rows = []
for regime, pool in source_pools.items():
    threshold = pool["node_hours"].quantile(
        HIGH_JOB_THRESHOLD_QUANTILE, interpolation="linear"
    )
    high_jobs = pool["node_hours"].gt(threshold)
    job_cutoff_rows.append(
        {
            "Regime": regime.title(),
            "Selected percentile": f"P{100 * HIGH_JOB_THRESHOLD_QUANTILE:.0f}",
            "Threshold [node-h]": threshold,
            "Jobs above threshold": int(high_jobs.sum()),
            "High jobs [%]": 100 * high_jobs.mean(),
        }
    )

job_cutoff_summary = pd.DataFrame(job_cutoff_rows).set_index("Regime")
display(job_cutoff_summary.round(2))

### 2. Require enough user history

A high-job share based on only a few jobs is unstable. This chart shows what remains as the minimum-history requirement increases: eligible source users, their share of replay jobs, and their share of total node-hours.

Moving right gives stronger evidence about each user's typical behavior, but excludes more users from heavy-user classification. The excluded users are still available for resampling; they simply keep the ordinary sampling weight.

In [ ]:
minimum_jobs_figure = mura.plot_minimum_jobs_decision(
    source_pools, figure_dir=FIGURE_DIR
)
display(minimum_jobs_figure)
plt.close(minimum_jobs_figure)


Set `MIN_JOBS_FOR_HEAVY` below. A user must have at least this many replay jobs in the regime to be eligible for heavy classification.

The table reports the exact number of eligible users and how much of the source workload they represent. Use it to check that the chosen minimum provides enough history without discarding most of the population from classification.

In [ ]:
# Classification choice 2: minimum evidence required from a source user.
MIN_JOBS_FOR_HEAVY = 15

eligibility_rows = []
for regime, pool in source_pools.items():
    job_counts = pool.groupby("source_user_ID").size()
    eligible = job_counts.ge(MIN_JOBS_FOR_HEAVY)
    eligibility_rows.append(
        {
            "Regime": regime.title(),
            "Minimum jobs": MIN_JOBS_FOR_HEAVY,
            "Source users": len(job_counts),
            "Eligible users": int(eligible.sum()),
            "Eligible users [%]": 100 * eligible.mean(),
            "Replay jobs from eligible users [%]": (
                100 * job_counts.loc[eligible].sum() / job_counts.sum()
            ),
        }
    )

eligibility_summary = pd.DataFrame(eligibility_rows).set_index("Regime")
display(eligibility_summary.round(2))

### 3. Choose the heavy tail among eligible users

For every eligible user, compute

$$
\text{high-job share}_u = \frac{\text{high-node-hour jobs submitted by }u}{\text{replay jobs submitted by }u}.
$$

The upper row shows the expected long-job and wide-job shares after applying each candidate heavy-user percentile and sampling weight. The dashed line is the original regime. The lower row shows how many eligible users would be labeled heavy at the same percentile.

In [ ]:
calibration_profiles = {
    regime: mura.measure_user_activity(
        pool,
        job_node_hours_quantile=HIGH_JOB_THRESHOLD_QUANTILE,
        min_jobs_for_heavy=MIN_JOBS_FOR_HEAVY,
    )
    for regime, pool in source_pools.items()
}

heavy_user_decision = mura.plot_heavy_user_decision(
    source_pools,
    calibration_profiles,
    SCENARIO_WEIGHTS,
    figure_dir=FIGURE_DIR,
)
display(heavy_user_decision)
plt.close(heavy_user_decision)


Set `HEAVY_USER_THRESHOLD_QUANTILE` below. Among eligible users, a user is heavy when their high-job share is greater than or equal to the regime-specific quantile.

The first table reports the actual classification. The second shows the expected workload composition under each sampling scenario before any random draw is made. `Effective source users` is $1 / \sum_u p_u^2$: it summarizes how concentrated the sampling probabilities are, with smaller values indicating that fewer source users dominate the draws.

In [ ]:
# Classification choice 3: upper tail of consistent high-job submitters.
HEAVY_USER_THRESHOLD_QUANTILE = 0.50

calibrated_users = {
    regime: mura.characterize_users(
        pool,
        job_node_hours_quantile=HIGH_JOB_THRESHOLD_QUANTILE,
        min_jobs_for_heavy=MIN_JOBS_FOR_HEAVY,
        heavy_user_threshold_quantile=HEAVY_USER_THRESHOLD_QUANTILE,
    )
    for regime, pool in source_pools.items()
}

calibrated_profile_table = pd.concat(
    [users.reset_index().assign(regime=regime) for regime, users in calibrated_users.items()],
    ignore_index=True,
)
classification_summary = mura.summarize_user_classification(calibrated_profile_table)
display(mura.format_user_classification(classification_summary))

selected_impact = pd.concat(
    [
        mura.expected_resampling_metrics(
            source_pools[regime], users,
            HEAVY_USER_THRESHOLD_QUANTILE, SCENARIO_WEIGHTS,
        ).assign(regime=regime)
        for regime, users in calibration_profiles.items()
    ],
    ignore_index=True,
)
display(mura.format_expected_resampling_metrics(selected_impact))

## Translate heavy labels into sampling probabilities

Every source user remains in the sampling population. In a heavy-weight scenario, heavy users receive weight $w$ and all other users receive weight 1:

$$
p_u = \frac{w_u}{\sum_v w_v}.
$$

Each synthetic workload draws exactly as many users as were active in the original regime, with replacement. A selected user contributes their complete replay subtrace. If the same source user is drawn twice, the two copies receive different synthetic user IDs.

The table below describes the expected mixture before sampling.

In [ ]:
prepared = {}
expected_rows = []
profile_tables = []

for regime, bounds in REGIMES.items():
    reference = json.loads((WORKLOAD_DIR / f"mustang_{regime}.json").read_text())
    pool, context = mura.prepare_regime(source, reference, bounds)
    users = mura.characterize_users(
        pool,
        job_node_hours_quantile=HIGH_JOB_THRESHOLD_QUANTILE,
        min_jobs_for_heavy=MIN_JOBS_FOR_HEAVY,
        heavy_user_threshold_quantile=HEAVY_USER_THRESHOLD_QUANTILE,
    )
    pd.testing.assert_series_equal(
        users["is_heavy"].sort_index(),
        calibrated_users[regime]["is_heavy"].sort_index(),
    )
    prepared[regime] = (pool, context, users, reference)
    profile_tables.append(users.reset_index().assign(regime=regime))
    for scenario, weight in SCENARIO_WEIGHTS.items():
        weights = np.where(users["is_heavy"], weight, 1.0)
        expected_rows.append(
            {
                "regime": regime,
                "scenario": scenario,
                "expected_heavy_user_share": np.average(users["is_heavy"], weights=weights),
                "expected_jobs": len(users) * np.average(users["jobs"], weights=weights),
                "expected_node_hours": len(users) * np.average(
                    users["total_node_hours"], weights=weights
                ),
            }
        )

source_user_profiles = pd.concat(profile_tables, ignore_index=True)
expected_composition = pd.DataFrame(expected_rows)
display(mura.format_expected_composition(expected_composition))

## Load or generate the collection

The saved collection contains 42 CSV/JSON workload pairs. Each CSV retains the nine Mustang columns and contains the four-week replay jobs. Each Batsim JSON uses 1,600 resources and includes the original context jobs unchanged. Stable sorting preserves the order of jobs with equal submission times.

With generation enabled, the cell writes the workload files, `manifest.csv`, `summary.csv`, `user_profiles.csv`, and generation metadata. It stops if validation finds invalid jobs, duplicate IDs, missing profiles, incomplete provenance, or a baseline that differs from the original Mustang workload.

With generation disabled, the cell checks the saved parameters and file hashes, then loads the collection.

In [ ]:
if GENERATE_WORKLOADS:
    summary, manifest, user_profiles, replays = mura.generate_artifacts(
        source=source,
        workload_dir=WORKLOAD_DIR,
        output_dir=ARTIFACT_DIR,
        regimes=REGIMES,
        job_node_hours_quantile=HIGH_JOB_THRESHOLD_QUANTILE,
        min_jobs_for_heavy=MIN_JOBS_FOR_HEAVY,
        heavy_user_threshold_quantile=HEAVY_USER_THRESHOLD_QUANTILE,
        scenario_weights=SCENARIO_WEIGHTS,
        seeds=SEEDS,
        source_path=SOURCE_CSV,
        hygiene=hygiene,
    )
else:
    metadata = json.loads((ARTIFACT_DIR / "metadata.json").read_text())
    expected_parameters = {
        "job_node_hours_quantile": HIGH_JOB_THRESHOLD_QUANTILE,
        "min_jobs_for_heavy": MIN_JOBS_FOR_HEAVY,
        "heavy_user_threshold_quantile": HEAVY_USER_THRESHOLD_QUANTILE,
        "scenario_weights": SCENARIO_WEIGHTS,
        "seeds": SEEDS,
        "regimes": {
            name: [value.isoformat() for value in mura.utc_window(bounds)]
            for name, bounds in REGIMES.items()
        },
    }
    for key, expected in expected_parameters.items():
        mura.require(
            metadata[key] == expected,
            f"Saved {key} differs. Restore the saved setting or generate a new collection in a new directory.",
        )
    summary, manifest, user_profiles, replays = mura.load_artifacts(ARTIFACT_DIR)

inventory = (
    summary.groupby(["regime", "scenario"], sort=False)
    .size()
    .rename("workloads")
    .unstack(fill_value=0)
)
display(inventory)
print(f"{'Generated' if GENERATE_WORKLOADS else 'Loaded'} {len(summary)} CSV/JSON workload pairs.")

## Verify the saved collection

This step reloads the CSV and JSON files. It checks the job records, profile references, context jobs, baseline equivalence, seeds, and provenance.

It also reconstructs each user draw in memory from the saved seed and weights. The comparison uses the persisted job attributes and user labels. This checks that the result is unchanged when the loop order is reversed.

In [ ]:
verification = mura.verify_artifacts(ARTIFACT_DIR, WORKLOAD_DIR)
display(pd.Series(verification, name="verified").to_frame())

for regime, (pool, _, users, _) in prepared.items():
    for scenario, weight in reversed(list(SCENARIO_WEIGHTS.items())):
        for seed in reversed(SEEDS):
            replay_again, _ = mura.resample(pool, users, regime, scenario, weight, seed)
            stem = f"mustang_{regime}_{scenario}_seed{seed}"
            persisted_columns = mura.CSV_COLUMNS + ["source_user_ID", "draw_order", "is_heavy"]
            pd.testing.assert_frame_equal(
                replay_again[persisted_columns], replays[stem][persisted_columns]
            )

print("All stochastic workloads reproduced exactly in reverse execution order.")

## Compare the generated workloads

The summary table first reports the mean and standard deviation across the five replicas in each scenario. Use the interactive views below to inspect individual workloads.

The overview is one bubble chart with a Slack/Stress selector. Each point is one workload: x is the long-job share, y is the wide-job share, bubble area increases with total node-hours, and color identifies the sampling scenario. Hover over a point to see its workload ID, seed, heavy-user draws, source concentration, and hourly-arrival correlation.

The distribution dashboard has one selector for all 42 workload IDs. It overlays the selected workload and the matching original baseline in five synchronized views: runtime, node count, daily submissions, synthetic-instance concentration, and source-user concentration. Hover and zoom are available in every panel.

In [ ]:
display(mura.format_collection_summary(summary))
display(mura.format_manifest_preview(manifest))

trace_overview = mura.plot_interactive_trace_overview(summary)
display(trace_overview)

distribution_dashboard = mura.plot_interactive_trace_distributions(summary, replays)
display(distribution_dashboard)


## Check the arrival pattern and choose a candidate

For each workload, align its 672 hourly submission counts with the original regime and compute their Pearson correlation:

$$
r = \operatorname{corr}(\text{candidate hourly counts}, \text{original hourly counts}).
$$

A value near 1 means that busy and quiet hours occur in a similar pattern.

The table lists every workload ID and its correlation. Use the dropdown in the hourly chart to overlay one candidate on its original baseline; hover for exact counts, zoom into a time interval, or use the range slider.

In [ ]:
hourly_correlations = mura.format_hourly_submission_correlations(summary)
display(hourly_correlations)

hourly_submission_figure = mura.plot_interactive_hourly_submissions(summary, replays)
display(hourly_submission_figure)


## Export one workload per regime for Greenfilling

Choose exactly two generated workloads: one Slack workload and one Stress workload. Then:

1. copy their IDs into the matching entries of `SELECTED_WORKLOAD_IDS`;
2. set `EXPORT_SELECTED_WORKLOADS = True`; and
3. run the export cell.

Both IDs are validated before either workload is exported. Each workload gets its own directory containing the CSV, Batsim JSON, user manifest, `selection.json`, and `comparison.png`. The PNG compares only that workload with the original Mustang trace from the same regime. It includes an explicit long/wide job-share comparison with percentage-point differences, plus runtime, node-count, daily-submission, hourly-submission, and user-concentration views.

Export is disabled by default and does not modify any campaign configuration.

In [ ]:
GREENFILLING_EXPORT_ROOT = (
    WORKLOAD_ROOT / "resampled"
)

# Copy one generated ID per regime from the table or Plotly dropdowns above.
SELECTED_WORKLOAD_IDS = {
    "slack": "mustang_slack_heavy_4x_seed20260930",
    "stress": "mustang_stress_heavy_8x_seed20260930",
}
EXPORT_SELECTED_WORKLOADS = False

if EXPORT_SELECTED_WORKLOADS:
    selected_exports = mura.export_selected_workload_pair(
        summary=summary,
        manifest=manifest,
        artifact_dir=ARTIFACT_DIR,
        selection_root=GREENFILLING_EXPORT_ROOT,
        workload_ids=SELECTED_WORKLOAD_IDS,
    )
    export_rows = [
        {
            "Regime": regime.title(),
            **{key: str(value) for key, value in exported.items()},
        }
        for regime, exported in selected_exports.items()
    ]
    display(pd.DataFrame(export_rows).set_index("Regime"))
else:
    print(
        "No workloads exported. Choose one generated Slack ID and one "
        "generated Stress ID, then set EXPORT_SELECTED_WORKLOADS = True."
    )

## Interpretation

The collection contains two original baselines and forty synthetic candidates. Each selected user contributes a complete copy of their replay jobs. Repeated draws give the copies separate synthetic user IDs.

The reported node-hours measure offered work. A larger value can increase congestion even when the arrival pattern remains similar. Hourly correlation measures the timing of submissions and does not measure their volume. Thus, the selected resampled campaign changes both job composition and offered load. The names slack and stress identify the source windows.

The five replicas describe variation across user draws. The existing campaign uses one selected replica per regime; its simulation results do not measure variation across seeds. Confidence intervals for that variation require simulations of multiple replicas.

The selected inputs remain in `workloads/resampled/`. Each directory includes the CSV, Batsim JSON, user manifest, selection record, and comparison figure.